## Init

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import avg, max, min, sum, round as spark_round, to_date, col

## Read from silver table

In [0]:
df_silver_read = spark.read.table("weather.weather_hourly")

## Aggregation: Daily summary by city

In [0]:
df_gold = (
    df_silver_read
    .withColumn("ingested_date", to_date(col("measurement_timestamp")))
    .groupBy("ingested_date", "state_code")
    .agg(
        spark_round(avg("temperature_celsius"), 2).alias("avg_temperature"),
        spark_round(max("temperature_celsius"), 2).alias("max_temperature"),
        spark_round(min("temperature_celsius"), 2).alias("min_temperature"),
        spark_round(avg("humidity_percentage"), 2).alias("avg_humidity"),
        spark_round(sum("precipitation_mm"), 2).alias("total_precipitation_mm")
    )
)

## Write data in gold table 
Using MERGE with (state_code + ingested_date) as unique key 

In [ ]:
table_name = "weather.daily_city_summary"

if not spark.catalog.tableExists(table_name):
    (
        df_gold
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
else: 
    target_table = DeltaTable.forName(spark, table_name)

    (
        target_table.alias("target")
        .merge(
            df_gold.alias("source"), 
            condition="""
            target.state_code = source.state_code AND 
            to_date(target.ingested_date) = to_date(source.ingested_date)
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

## Sanity check

In [ ]:
%sql
select * 
from weather.daily_city_summary
order by ingested_date desc, state_code
limit 10